# Technical Note: Density Matrix and Entanglement Evaluation under Noise

This notebook summarizes:
1. The concept of the density matrix
2. How to calculate density matrices under noise
3. Entanglement evaluation methodology
   - von Neumann entropy
   - Concurrence
   - Monogamy (CKW inequality)


## 0) Setup


In [ ]:
import numpy as np
from numpy import kron

from qiskit.quantum_info import (
    Statevector,
    DensityMatrix,
    partial_trace,
    entropy,
)


## 1) Concept of Density Matrix

A density matrix $\rho$ represents a quantum state in a unified way for both pure and mixed states.

- **Pure state**: $\rho = |\psi\rangle\langle\psi|$
- **Mixed state**: $\rho = \sum_i p_i |\psi_i\rangle\langle\psi_i|$, with $\sum_i p_i = 1$

Physical constraints:
- Hermitian: $\rho = \rho^\dagger$
- Positive semidefinite: eigenvalues $\lambda_i \ge 0$
- Unit trace: $\mathrm{Tr}(\rho)=1$

For multipartite systems, reduced states are obtained by partial trace, e.g.
$$\rho_A = \mathrm{Tr}_B(\rho_{AB}).$$

In [ ]:
# Bell state |Phi+> = (|00> + |11>) / sqrt(2)
bell = Statevector.from_label('00').evolve([[1/np.sqrt(2), 0, 0, 1/np.sqrt(2)],
                                              [0, 1, 0, 0],
                                              [0, 0, 1, 0],
                                              [1/np.sqrt(2), 0, 0, -1/np.sqrt(2)]])
rho_bell = DensityMatrix(bell)

print('rho shape:', rho_bell.data.shape)
print('trace(rho):', np.trace(rho_bell.data))
print('Hermitian check:', np.allclose(rho_bell.data, rho_bell.data.conj().T))
print('eigenvalues:', np.linalg.eigvalsh(rho_bell.data))


## 2) Density Matrix under Noise

A noisy channel $\mathcal{E}$ acting on density matrix $\rho$ is expressed by Kraus operators $\{E_k\}$:
$$\rho' = \mathcal{E}(\rho) = \sum_k E_k \rho E_k^\dagger,$$
with completeness condition $\sum_k E_k^\dagger E_k = I$.

If noise acts on one qubit inside a multi-qubit register, we embed each Kraus operator into the full Hilbert space using tensor products.


In [ ]:
def expand_single_qubit_op(op, target, n_qubits):
    """Expand a 2x2 operator to n qubits, acting on `target`."""
    mats = []
    for q in range(n_qubits):
        mats.append(op if q == target else np.eye(2, dtype=complex))
    full = mats[0]
    for m in mats[1:]:
        full = kron(full, m)
    return full

def apply_local_kraus(rho, kraus_ops_1q, target, n_qubits):
    rho_new = np.zeros_like(rho, dtype=complex)
    for k in kraus_ops_1q:
        K = expand_single_qubit_op(k, target, n_qubits)
        rho_new += K @ rho @ K.conj().T
    return rho_new

def amplitude_damping_kraus(gamma):
    E0 = np.array([[1, 0], [0, np.sqrt(1 - gamma)]], dtype=complex)
    E1 = np.array([[0, np.sqrt(gamma)], [0, 0]], dtype=complex)
    return [E0, E1]

# Initial 2-qubit Bell state density matrix
psi_bell = (Statevector.from_label('00') + Statevector.from_label('11')) / np.sqrt(2)
rho0 = DensityMatrix(psi_bell).data

# Apply amplitude damping to qubit 0
gamma = 0.2
rho_noisy = apply_local_kraus(rho0, amplitude_damping_kraus(gamma), target=0, n_qubits=2)

print('trace(rho_noisy):', np.trace(rho_noisy))
print('min eigenvalue:', np.min(np.linalg.eigvalsh(rho_noisy)).real)


## 3) Methodology of Entanglement Evaluations

### 3.1 von Neumann Entropy

For state $\rho$:
$$S(\rho) = -\mathrm{Tr}(\rho \log_2 \rho).$$

For a bipartite pure state $|\psi\rangle_{AB}$, entanglement is quantified by:
$$E(|\psi\rangle_{AB}) = S(\rho_A) = S(\rho_B), \quad \rho_A = \mathrm{Tr}_B(|\psi\rangle\langle\psi|).$$

Under noise, evaluate entropy on reduced density matrices after applying the channel.


In [ ]:
def entanglement_entropy_qubitA(rho_2q):
    rho_dm = DensityMatrix(rho_2q)
    # Trace out qubit 1 -> reduced state of qubit 0
    rho_A = partial_trace(rho_dm, [1])
    return entropy(rho_A, base=2)

gammas = np.linspace(0, 1, 6)
print('gamma | S(rho_A)')
for g in gammas:
    rho_g = apply_local_kraus(rho0, amplitude_damping_kraus(g), target=0, n_qubits=2)
    sA = entanglement_entropy_qubitA(rho_g)
    print(f'{g:>4.2f}  | {sA:>7.4f}')


### 3.2 Concurrence

For a two-qubit state $\rho$, Wootters concurrence is:
$$C(\rho)=\max\left(0, \lambda_1-\lambda_2-\lambda_3-\lambda_4\right),$$
where $\lambda_i$ are square roots (descending) of eigenvalues of
$$R = \rho(\sigma_y\otimes\sigma_y)\rho^*(\sigma_y\otimes\sigma_y).$$

Range: $0$ (separable) to $1$ (maximally entangled).


In [ ]:
def concurrence_2qubit(rho_2q):
    sy = np.array([[0, -1j], [1j, 0]], dtype=complex)
    YY = kron(sy, sy)
    R = rho_2q @ YY @ rho_2q.conj() @ YY

    eigvals = np.linalg.eigvals(R)
    eigvals = np.real_if_close(eigvals)
    eigvals = np.maximum(np.real(eigvals), 0.0)
    lambdas = np.sort(np.sqrt(eigvals))[::-1]
    return max(0.0, lambdas[0] - lambdas[1] - lambdas[2] - lambdas[3])

print('gamma | concurrence')
for g in gammas:
    rho_g = apply_local_kraus(rho0, amplitude_damping_kraus(g), target=0, n_qubits=2)
    c = concurrence_2qubit(rho_g)
    print(f'{g:>4.2f}  | {c:>9.6f}')


### 3.3 Monogamy of Entanglement

For a three-qubit pure state $|\psi\rangle_{ABC}$, the Coffman-Kundu-Wootters (CKW) monogamy relation is:
$$C_{A|BC}^2 \ge C_{AB}^2 + C_{AC}^2.$$

- $C_{AB}$ and $C_{AC}$ are pairwise concurrences.
- $C_{A|BC}$ is concurrence across bipartition $A$ vs $(BC)$.

Residual tangle:
$$\tau_A = C_{A|BC}^2 - C_{AB}^2 - C_{AC}^2 \ge 0.$$

Interpretation:
- GHZ-like states: strong genuine tripartite entanglement ($\tau_A > 0$).
- W-like states: mostly pairwise sharing ($\tau_A \approx 0$).


In [ ]:
def reduced_density(rho, trace_out):
    return partial_trace(DensityMatrix(rho), trace_out).data

def c_a_bc_for_pure_3q(rho_abc):
    # For pure 3-qubit state: C_{A|BC} = 2 * sqrt(det(rho_A))
    rho_A = reduced_density(rho_abc, [1, 2])
    detA = np.linalg.det(rho_A)
    val = 2 * np.sqrt(max(0.0, np.real_if_close(detA).real))
    return float(np.real(val))

def monogamy_metrics(statevec_3q):
    rho_abc = DensityMatrix(statevec_3q).data

    rho_ab = reduced_density(rho_abc, [2])
    rho_ac = reduced_density(rho_abc, [1])

    c_ab = concurrence_2qubit(rho_ab)
    c_ac = concurrence_2qubit(rho_ac)
    c_a_bc = c_a_bc_for_pure_3q(rho_abc)

    lhs = c_a_bc**2
    rhs = c_ab**2 + c_ac**2
    tau = lhs - rhs
    return c_a_bc, c_ab, c_ac, lhs, rhs, tau

# GHZ and W states
ghz = (Statevector.from_label('000') + Statevector.from_label('111')) / np.sqrt(2)
w = (Statevector.from_label('001') + Statevector.from_label('010') + Statevector.from_label('100')) / np.sqrt(3)

for name, st in [('GHZ', ghz), ('W', w)]:
    c_a_bc, c_ab, c_ac, lhs, rhs, tau = monogamy_metrics(st)
    print(f'[{name}]')
    print(f'  C_A|BC = {c_a_bc:.6f}')
    print(f'  C_AB   = {c_ab:.6f}')
    print(f'  C_AC   = {c_ac:.6f}')
    print(f'  LHS    = C_A|BC^2 = {lhs:.6f}')
    print(f'  RHS    = C_AB^2 + C_AC^2 = {rhs:.6f}')
    print(f'  tau_A  = {tau:.6f}')
    print()


## 4) Notes and Practical Guidance

- For noisy simulations, always verify physicality of $\rho$ (Hermitian, PSD, trace 1).
- Entropy and concurrence can respond differently under different channels.
- Monogamy helps distinguish pairwise entanglement from genuine multipartite correlations.
- In experiments, report channel assumptions (Kraus operators, noise strength, target qubits) for reproducibility.
